# 🎬 CineAI — Intelligent Movie Recommendation System
### A Production-Grade Deep Learning Project

---

**Architecture:** Hybrid LSTM + Transformer with Attention Mechanisms  
**Dataset:** MovieLens 100K (943 users · 1,682 movies · 100,000 ratings)  
**Models:** Sequential LSTM · Self-Attention Transformer · Hybrid Ensemble  
**UI:** Professional Gradio interface with analytics dashboard  

---

> **Project Modules:**
> 1. Environment Setup & GPU Verification
> 2. Data Pipeline & Exploratory Analysis  
> 3. Feature Engineering & Preprocessing  
> 4. LSTM Model Architecture & Training  
> 5. Transformer Model Architecture & Training  
> 6. Hybrid Ensemble Model  
> 7. Model Evaluation & Benchmarking  
> 8. Recommendation Engine  
> 9. Professional Gradio UI Launch


## 📦 Module 1 — Environment Setup & GPU Verification

In [ ]:
# ── Install all required packages ──────────────────────────────
!pip install gradio plotly scikit-learn tensorflow --quiet

import os, sys, time, warnings, urllib.request, zipfile
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                              precision_score, recall_score)

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, GRU, Dense, Dropout, BatchNormalization,
    MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D,
    Conv1D, Flatten, Concatenate, Add, Multiply, Reshape,
    Bidirectional, Attention
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, LambdaCallback
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
import tensorflow.keras.backend as K

# ── GPU Check ──────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
print('='*60)
print('  CINEAI ENVIRONMENT REPORT')
print('='*60)
print(f'  Python        : {sys.version.split()[0]}')
print(f'  TensorFlow    : {tf.__version__}')
print(f'  NumPy         : {np.__version__}')
print(f'  Pandas        : {pd.__version__}')
print(f'  GPU Available : {"YES ✅" if gpus else "NO ❌"}')
if gpus:
    for g in gpus:
        print(f'  GPU Device    : {g.name}')
print('='*60)

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('\n✅ All libraries loaded. Environment ready.')


## 📊 Module 2 — Data Pipeline & Exploratory Analysis

In [ ]:
# ── Download & Extract MovieLens 100K ─────────────────────────
print('Downloading MovieLens 100K dataset...')
urllib.request.urlretrieve(
    'https://files.grouplens.org/datasets/movielens/ml-100k.zip',
    'ml-100k.zip'
)
with zipfile.ZipFile('ml-100k.zip', 'r') as z:
    z.extractall('.')
print('✅ Dataset downloaded.')

# ── Load ratings ───────────────────────────────────────────────
ratings = pd.read_csv('ml-100k/u.data', sep='\t',
    names=['user_id','movie_id','rating','timestamp'])

# ── Load movies ────────────────────────────────────────────────
genre_cols = ['unknown','Action','Adventure','Animation','Children',
              'Comedy','Crime','Documentary','Drama','Fantasy',
              'Film-Noir','Horror','Musical','Mystery','Romance',
              'Sci-Fi','Thriller','War','Western']
m_cols = ['movie_id','title','release_date','video_release_date',
          'imdb_url'] + genre_cols
movies = pd.read_csv('ml-100k/u.item', sep='|', names=m_cols,
                     encoding='latin-1')

# ── Load users ─────────────────────────────────────────────────
users = pd.read_csv('ml-100k/u.user', sep='|',
    names=['user_id','age','gender','occupation','zip_code'])

print('='*60)
print('  DATASET SUMMARY')
print('='*60)
print(f'  Users         : {users.shape[0]:,}')
print(f'  Movies        : {movies.shape[0]:,}')
print(f'  Ratings       : {ratings.shape[0]:,}')
print(f'  Rating scale  : {ratings.rating.min()} – {ratings.rating.max()}')
print(f'  Sparsity      : {(1 - len(ratings)/(users.shape[0]*movies.shape[0]))*100:.2f}%')
print(f'  Avg rating    : {ratings.rating.mean():.3f}')
print(f'  Date range    : {pd.to_datetime(ratings.timestamp,unit="s").dt.year.min()} – '
      f'{pd.to_datetime(ratings.timestamp,unit="s").dt.year.max()}')
print('='*60)


In [ ]:
# ── Exploratory Data Analysis ─────────────────────────────────
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Rating Distribution', 'Ratings per User (top 30)',
        'Ratings per Movie (top 30)', 'Genre Popularity',
        'Rating Trend Over Time', 'Age vs Avg Rating'
    ]
)

# 1. Rating distribution
rating_counts = ratings['rating'].value_counts().sort_index()
fig.add_trace(go.Bar(
    x=rating_counts.index.astype(str),
    y=rating_counts.values,
    marker_color=['#e50914' if i==4 else '#444' for i in range(5)],
    name='Ratings'
), row=1, col=1)

# 2. Top users
top_users = ratings.groupby('user_id').size().sort_values(ascending=False).head(30)
fig.add_trace(go.Bar(
    x=list(range(30)), y=top_users.values,
    marker_color='#e50914', name='Top Users'
), row=1, col=2)

# 3. Top movies
top_movies = ratings.groupby('movie_id').size().sort_values(ascending=False).head(30)
fig.add_trace(go.Bar(
    x=list(range(30)), y=top_movies.values,
    marker_color='#cc0000', name='Top Movies'
), row=1, col=3)

# 4. Genre popularity
genre_counts = movies[genre_cols[1:]].sum().sort_values(ascending=False)
fig.add_trace(go.Bar(
    x=genre_counts.values[:10],
    y=genre_counts.index[:10],
    orientation='h',
    marker_color='#e50914', name='Genres'
), row=2, col=1)

# 5. Monthly trend
ratings['date'] = pd.to_datetime(ratings['timestamp'], unit='s')
monthly = ratings.set_index('date').resample('M')['rating'].mean()
fig.add_trace(go.Scatter(
    x=monthly.index, y=monthly.values,
    line=dict(color='#e50914', width=2),
    name='Avg Rating'
), row=2, col=2)

# 6. Age vs rating
merged = ratings.merge(users, on='user_id')
age_rating = merged.groupby('age')['rating'].mean().reset_index()
fig.add_trace(go.Scatter(
    x=age_rating['age'], y=age_rating['rating'],
    mode='markers+lines',
    marker=dict(color='#e50914', size=6),
    name='Age vs Rating'
), row=2, col=3)

fig.update_layout(
    height=700, showlegend=False,
    paper_bgcolor='#ffffff', plot_bgcolor='#f8f9fb',
    font=dict(color='#1a1a2e', size=11),
    title=dict(
        text='📊 CineAI — Exploratory Data Analysis Dashboard',
        font=dict(size=18, color='#c0392b')
    )
)
fig.update_xaxes(gridcolor='#dde1ea', showgrid=True)
fig.update_yaxes(gridcolor='#dde1ea', showgrid=True)
fig.show()
print('✅ EDA complete.')


## ⚙️ Module 3 — Feature Engineering & Preprocessing

In [ ]:
# ── 1. User-Movie interaction matrix ──────────────────────────
print('Building interaction matrix...')
interaction_matrix = ratings.pivot_table(
    index='user_id', columns='movie_id',
    values='rating', fill_value=0
)

# ── 2. Genre feature vectors ───────────────────────────────────
genre_features = movies.set_index('movie_id')[genre_cols[1:]].values.astype(np.float32)

# ── 3. Normalize ratings ───────────────────────────────────────
ratings['rating_norm'] = (ratings['rating'] - 1) / 4.0   # scale 1-5 → 0-1

# ── 4. User statistics features ────────────────────────────────
user_stats = ratings.groupby('user_id')['rating'].agg(
    user_mean='mean', user_std='std', user_count='count'
).fillna(0).reset_index()

# ── 5. Movie statistics features ───────────────────────────────
movie_stats = ratings.groupby('movie_id')['rating'].agg(
    movie_mean='mean', movie_std='std', movie_count='count'
).fillna(0).reset_index()

# ── 6. Merge all features ──────────────────────────────────────
df = ratings.merge(user_stats, on='user_id').merge(movie_stats, on='movie_id')
df = df.merge(movies[['movie_id','title']+genre_cols[1:]], on='movie_id')
df = df.merge(users[['user_id','age','gender','occupation']], on='user_id')

# Encode categoricals
le_gender = LabelEncoder()
le_occ    = LabelEncoder()
df['gender_enc']     = le_gender.fit_transform(df['gender'])
df['occupation_enc'] = le_occ.fit_transform(df['occupation'])

# ── 7. Sequence data (for LSTM/Transformer) ────────────────────
print('Building watch sequences...')
liked = ratings[ratings['rating'] >= 3].sort_values('timestamp')
user_sequences = liked.groupby('user_id')['movie_id'].apply(list).to_dict()

WINDOW = 5
X_seq, y_seq = [], []
for uid, hist in user_sequences.items():
    if len(hist) > WINDOW:
        for i in range(WINDOW, len(hist)):
            X_seq.append(hist[i-WINDOW:i])
            y_seq.append(hist[i])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=SEED
)

# ── 8. Collaborative filtering data ────────────────────────────
X_cf = df[['user_id','movie_id']].values
y_cf = df['rating_norm'].values
X_train_cf, X_test_cf, y_train_cf, y_test_cf = train_test_split(
    X_cf, y_cf, test_size=0.2, random_state=SEED
)

NUM_USERS  = ratings['user_id'].max() + 1
NUM_MOVIES = ratings['movie_id'].max() + 1
EMBED_DIM  = 64

print('='*60)
print('  FEATURE ENGINEERING SUMMARY')
print('='*60)
print(f'  Interaction matrix : {interaction_matrix.shape}')
print(f'  Sequence samples   : {len(X_seq):,}')
print(f'  Train sequences    : {len(X_train_s):,}')
print(f'  Test sequences     : {len(X_test_s):,}')
print(f'  CF train samples   : {len(X_train_cf):,}')
print(f'  Vocab size         : {NUM_MOVIES:,}')
print(f'  Window size        : {WINDOW}')
print('='*60)
print('✅ Feature engineering complete.')


## 🧠 Module 4 — LSTM Model Architecture & Training

In [ ]:
# ── Bidirectional LSTM with Attention ─────────────────────────

class AttentionLayer(tf.keras.layers.Layer):
    """Custom Bahdanau-style attention mechanism."""
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.W = Dense(units, use_bias=False)
        self.V = Dense(1, use_bias=False)

    def call(self, hidden_states):
        score  = self.V(tf.nn.tanh(self.W(hidden_states)))
        weights = tf.nn.softmax(score, axis=1)
        context = weights * hidden_states
        return tf.reduce_sum(context, axis=1), weights

def build_lstm_model(vocab_size, embed_dim, seq_len, num_units=128):
    inp = Input(shape=(seq_len,), name='sequence_input')

    # Embedding with L2 regularisation
    emb = Embedding(vocab_size, embed_dim,
                    embeddings_regularizer=l2(1e-5),
                    name='item_embedding')(inp)
    emb = Dropout(0.2)(emb)

    # Bidirectional LSTM stack
    x = Bidirectional(LSTM(num_units, return_sequences=True,
                            dropout=0.2, recurrent_dropout=0.1,
                            name='lstm_1'))(emb)
    x = BatchNormalization()(x)

    x = Bidirectional(LSTM(num_units//2, return_sequences=True,
                            dropout=0.2, name='lstm_2'))(x)
    x = BatchNormalization()(x)

    # Custom attention
    attn_layer = AttentionLayer(units=64)
    context, attn_weights = attn_layer(x)

    # Dense head
    x = Dense(256, activation='relu', kernel_regularizer=l2(1e-4))(context)
    x = Dropout(0.3)(x)
    x = BatchNormalization()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.2)(x)
    out = Dense(vocab_size, activation='softmax', name='output')(x)

    model = Model(inp, out, name='CineAI_LSTM')
    model.compile(
        optimizer=Adam(learning_rate=5e-4, clipnorm=1.0),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5_acc'),
                 tf.keras.metrics.SparseTopKCategoricalAccuracy(k=10, name='top10_acc')]
    )
    return model

lstm_model = build_lstm_model(NUM_MOVIES, EMBED_DIM, WINDOW)
lstm_model.summary()
print(f'\nTotal parameters: {lstm_model.count_params():,}')


In [ ]:
# ── Train LSTM ────────────────────────────────────────────────
lstm_callbacks = [
    EarlyStopping(monitor='val_loss', patience=4,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_lstm.keras', save_best_only=True,
                    monitor='val_top5_acc', mode='max', verbose=0),
]

print('🚀 Training LSTM model...')
t0 = time.time()
lstm_history = lstm_model.fit(
    X_train_s, y_train_s,
    epochs=15,
    batch_size=128,
    validation_data=(X_test_s, y_test_s),
    callbacks=lstm_callbacks,
    verbose=1
)
lstm_time = time.time() - t0

lstm_acc    = max(lstm_history.history['val_accuracy'])
lstm_top5   = max(lstm_history.history['val_top5_acc'])
lstm_top10  = max(lstm_history.history['val_top10_acc'])
lstm_loss   = min(lstm_history.history['val_loss'])

print(f'\n✅ LSTM training complete in {lstm_time/60:.1f} minutes')
print(f'   Best val accuracy : {lstm_acc:.4f}')
print(f'   Best top-5 acc    : {lstm_top5:.4f}')
print(f'   Best top-10 acc   : {lstm_top10:.4f}')
print(f'   Best val loss     : {lstm_loss:.4f}')


## ⚡ Module 5 — Transformer Architecture & Training

In [ ]:
# ── Transformer with Positional Encoding ──────────────────────

class PositionalEncoding(tf.keras.layers.Layer):
    """Sinusoidal positional encoding (Vaswani et al. 2017)."""
    def __init__(self, max_len=100, d_model=64, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.d_model = d_model

    def call(self, x):
        positions = np.arange(self.max_len)[:, np.newaxis]
        dims      = np.arange(self.d_model)[np.newaxis, :]
        angles    = positions / np.power(10000, (2*(dims//2)) / self.d_model)
        angles[:, 0::2] = np.sin(angles[:, 0::2])
        angles[:, 1::2] = np.cos(angles[:, 1::2])
        pos_enc   = tf.cast(angles[np.newaxis, :, :], dtype=tf.float32)
        seq_len   = tf.shape(x)[1]
        return x + pos_enc[:, :seq_len, :]

class TransformerBlock(tf.keras.layers.Layer):
    """Full Transformer encoder block with pre-LN."""
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attn  = MultiHeadAttention(num_heads=num_heads,
                                         key_dim=embed_dim//num_heads,
                                         dropout=dropout)
        self.ff1   = Dense(ff_dim, activation='gelu')
        self.ff2   = Dense(embed_dim)
        self.ln1   = LayerNormalization(epsilon=1e-6)
        self.ln2   = LayerNormalization(epsilon=1e-6)
        self.drop1 = Dropout(dropout)
        self.drop2 = Dropout(dropout)

    def call(self, x, training=False):
        # Pre-LN attention
        x_norm = self.ln1(x)
        attn_out = self.attn(x_norm, x_norm, training=training)
        x = x + self.drop1(attn_out, training=training)
        # Pre-LN FFN
        x_norm = self.ln2(x)
        ff_out = self.ff2(self.ff1(x_norm))
        x = x + self.drop2(ff_out, training=training)
        return x

def build_transformer_model(vocab_size, embed_dim, seq_len,
                             num_heads=8, ff_dim=256, num_blocks=4):
    inp = Input(shape=(seq_len,), name='sequence_input')

    # Token embedding + scale
    emb = Embedding(vocab_size, embed_dim,
                    embeddings_regularizer=l2(1e-5),
                    name='item_embedding')(inp)
    emb = emb * tf.math.sqrt(tf.cast(embed_dim, tf.float32))
    emb = Dropout(0.2)(emb)

    # Positional encoding
    x = PositionalEncoding(max_len=seq_len, d_model=embed_dim)(emb)

    # Stacked transformer blocks
    for i in range(num_blocks):
        x = TransformerBlock(embed_dim, num_heads, ff_dim,
                              dropout=0.1, name=f'transformer_block_{i}')(x)

    # Aggregate
    x = LayerNormalization(epsilon=1e-6)(x)
    x = GlobalAveragePooling1D()(x)

    # Classification head
    x = Dense(512, activation='gelu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.3)(x)
    x = Dense(256, activation='gelu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.2)(x)
    out = Dense(vocab_size, activation='softmax', name='output')(x)

    model = Model(inp, out, name='CineAI_Transformer')
    model.compile(
        optimizer=Adam(learning_rate=3e-4, clipnorm=1.0),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5_acc'),
                 tf.keras.metrics.SparseTopKCategoricalAccuracy(k=10, name='top10_acc')]
    )
    return model

transformer_model = build_transformer_model(
    NUM_MOVIES, EMBED_DIM, WINDOW,
    num_heads=8, ff_dim=256, num_blocks=4
)
transformer_model.summary()
print(f'\nTotal parameters: {transformer_model.count_params():,}')


In [ ]:
# ── Train Transformer ─────────────────────────────────────────
trans_callbacks = [
    EarlyStopping(monitor='val_loss', patience=4,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_transformer.keras', save_best_only=True,
                    monitor='val_top5_acc', mode='max', verbose=0),
]

print('🚀 Training Transformer model...')
t0 = time.time()
trans_history = transformer_model.fit(
    X_train_s, y_train_s,
    epochs=15,
    batch_size=128,
    validation_data=(X_test_s, y_test_s),
    callbacks=trans_callbacks,
    verbose=1
)
trans_time = time.time() - t0

trans_acc   = max(trans_history.history['val_accuracy'])
trans_top5  = max(trans_history.history['val_top5_acc'])
trans_top10 = max(trans_history.history['val_top10_acc'])
trans_loss  = min(trans_history.history['val_loss'])

print(f'\n✅ Transformer training complete in {trans_time/60:.1f} minutes')
print(f'   Best val accuracy : {trans_acc:.4f}')
print(f'   Best top-5 acc    : {trans_top5:.4f}')
print(f'   Best top-10 acc   : {trans_top10:.4f}')
print(f'   Best val loss     : {trans_loss:.4f}')


## 🔗 Module 6 — Neural Collaborative Filtering (NCF) Model

In [ ]:
# ── Neural Collaborative Filtering ────────────────────────────
# Combines matrix factorisation (GMF) + MLP for rating prediction

def build_ncf_model(num_users, num_movies, embed_dim=32):
    # Inputs
    user_input  = Input(shape=(1,), name='user_input')
    movie_input = Input(shape=(1,), name='movie_input')

    # ─ GMF branch (Generalised Matrix Factorisation) ─
    user_emb_gmf  = Embedding(num_users,  embed_dim, name='user_emb_gmf')(user_input)
    movie_emb_gmf = Embedding(num_movies, embed_dim, name='movie_emb_gmf')(movie_input)
    gmf = Multiply()([Flatten()(user_emb_gmf), Flatten()(movie_emb_gmf)])

    # ─ MLP branch ─
    user_emb_mlp  = Embedding(num_users,  embed_dim, name='user_emb_mlp')(user_input)
    movie_emb_mlp = Embedding(num_movies, embed_dim, name='movie_emb_mlp')(movie_input)
    mlp = Concatenate()([Flatten()(user_emb_mlp), Flatten()(movie_emb_mlp)])
    mlp = Dense(128, activation='relu')(mlp)
    mlp = Dropout(0.3)(mlp)
    mlp = Dense(64,  activation='relu')(mlp)
    mlp = Dropout(0.2)(mlp)
    mlp = Dense(32,  activation='relu')(mlp)

    # ─ Fusion ─
    x = Concatenate()([gmf, mlp])
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    out = Dense(1, activation='sigmoid', name='rating_output')(x)

    model = Model([user_input, movie_input], out, name='CineAI_NCF')
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['mae']
    )
    return model

ncf_model = build_ncf_model(NUM_USERS, NUM_MOVIES, embed_dim=32)
ncf_model.summary()
print(f'\nTotal parameters: {ncf_model.count_params():,}')

# Prepare CF data
X_u = X_train_cf[:, 0].reshape(-1,1)
X_m = X_train_cf[:, 1].reshape(-1,1)
Xu_t = X_test_cf[:, 0].reshape(-1,1)
Xm_t = X_test_cf[:, 1].reshape(-1,1)

print('\n🚀 Training NCF model...')
ncf_history = ncf_model.fit(
    [X_u, X_m], y_train_cf,
    validation_data=([Xu_t, Xm_t], y_test_cf),
    epochs=10, batch_size=256,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)
print('✅ NCF training complete.')


## 📈 Module 7 — Model Evaluation & Benchmarking

In [ ]:
# ── Comprehensive evaluation dashboard ─────────────────────────

def hit_rate_at_k(model, X_test, y_test, k=10, n_samples=2000):
    """Hit Rate @ K: fraction of test cases where true item is in top-K."""
    idx    = np.random.choice(len(X_test), min(n_samples, len(X_test)), replace=False)
    X_s, y_s = X_test[idx], y_test[idx]
    preds  = model.predict(X_s, verbose=0, batch_size=256)
    top_k  = np.argsort(preds, axis=1)[:, -k:]
    hits   = sum(y_s[i] in top_k[i] for i in range(len(y_s)))
    return hits / len(y_s)

def ndcg_at_k(model, X_test, y_test, k=10, n_samples=2000):
    """Normalised Discounted Cumulative Gain @ K."""
    idx   = np.random.choice(len(X_test), min(n_samples, len(X_test)), replace=False)
    X_s, y_s = X_test[idx], y_test[idx]
    preds = model.predict(X_s, verbose=0, batch_size=256)
    ndcg  = 0.0
    for i in range(len(y_s)):
        top_k = np.argsort(preds[i])[-k:][::-1].tolist()
        if y_s[i] in top_k:
            rank  = top_k.index(y_s[i]) + 1
            ndcg += 1.0 / np.log2(rank + 1)
    return ndcg / len(y_s)

print('Computing evaluation metrics (this may take a moment)...')

metrics = {}
for name, model in [('LSTM', lstm_model), ('Transformer', transformer_model)]:
    hr5  = hit_rate_at_k(model, X_test_s, y_test_s, k=5)
    hr10 = hit_rate_at_k(model, X_test_s, y_test_s, k=10)
    nd5  = ndcg_at_k(model, X_test_s, y_test_s, k=5)
    nd10 = ndcg_at_k(model, X_test_s, y_test_s, k=10)
    acc  = max((lstm_history if name=='LSTM' else trans_history).history['val_accuracy'])
    t5   = max((lstm_history if name=='LSTM' else trans_history).history['val_top5_acc'])
    t10  = max((lstm_history if name=='LSTM' else trans_history).history['val_top10_acc'])
    metrics[name] = dict(Accuracy=acc, Top5_Acc=t5, Top10_Acc=t10,
                         HR5=hr5, HR10=hr10, NDCG5=nd5, NDCG10=nd10)

# ── Print results table ────────────────────────────────────────
print('\n' + '='*65)
print(f'  {"Metric":<22} {"LSTM":>18} {"Transformer":>18}')
print('='*65)
for metric in ['Accuracy','Top5_Acc','Top10_Acc','HR5','HR10','NDCG5','NDCG10']:
    l = metrics['LSTM'][metric]
    t = metrics['Transformer'][metric]
    winner = '◀ LSTM' if l > t else 'Transformer ▶'
    print(f'  {metric:<22} {l:>17.4f}  {t:>17.4f}  ({winner})')
print('='*65)
winner_model = 'Transformer' if metrics['Transformer']['Top10_Acc'] > metrics['LSTM']['Top10_Acc'] else 'LSTM'
print(f'  Overall Winner: {winner_model} ✅')
print('='*65)


In [ ]:
# ── Training curve visualisation ──────────────────────────────
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Val Accuracy', 'Val Top-5 Accuracy', 'Val Top-10 Accuracy',
        'Training Loss', 'LSTM vs Transformer (HR@10)',
        'LSTM vs Transformer (NDCG@10)'
    ]
)
colors = {'LSTM': '#e50914', 'Transformer': '#f5c518'}

for name, hist in [('LSTM', lstm_history), ('Transformer', trans_history)]:
    c = colors[name]
    ep = list(range(1, len(hist.history['val_accuracy'])+1))
    fig.add_trace(go.Scatter(x=ep, y=hist.history['val_accuracy'],
        name=name, line=dict(color=c, width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=ep, y=hist.history['val_top5_acc'],
        name=name, line=dict(color=c, width=2), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=ep, y=hist.history['val_top10_acc'],
        name=name, line=dict(color=c, width=2), showlegend=False), row=1, col=3)
    fig.add_trace(go.Scatter(x=ep, y=hist.history['val_loss'],
        name=name, line=dict(color=c, width=2), showlegend=False), row=2, col=1)

# Bar: HR@10
fig.add_trace(go.Bar(
    x=['LSTM','Transformer'],
    y=[metrics['LSTM']['HR10'], metrics['Transformer']['HR10']],
    marker_color=[colors['LSTM'], colors['Transformer']], showlegend=False
), row=2, col=2)

# Bar: NDCG@10
fig.add_trace(go.Bar(
    x=['LSTM','Transformer'],
    y=[metrics['LSTM']['NDCG10'], metrics['Transformer']['NDCG10']],
    marker_color=[colors['LSTM'], colors['Transformer']], showlegend=False
), row=2, col=3)

fig.update_layout(
    height=600,
    paper_bgcolor='#ffffff', plot_bgcolor='#f8f9fb',
    font=dict(color='#1a1a2e'),
    title=dict(text='📈 CineAI — Model Evaluation Dashboard',
               font=dict(size=16, color='#c0392b')),
    legend=dict(bgcolor='#ffffff', bordercolor='#dde1ea', borderwidth=1)
)
fig.update_xaxes(gridcolor='#dde1ea')
fig.update_yaxes(gridcolor='#dde1ea')
fig.show()


## 🎯 Module 8 — Recommendation Engine

In [ ]:
# ── Production-grade recommendation engine ─────────────────────

# Build lookup tables
title_to_id = pd.Series(movies.movie_id.values, index=movies.title).to_dict()
id_to_title  = pd.Series(movies.title.values,    index=movies.movie_id.values).to_dict()
id_to_genres = {}
for _, row in movies.iterrows():
    genres = [g for g in genre_cols[1:] if row[g] == 1]
    id_to_genres[row['movie_id']] = genres

# All movie titles for UI
all_titles = sorted(movies['title'].unique().tolist())
genre_list = genre_cols[1:]

class RecommendationEngine:
    """
    Unified recommendation engine supporting:
    - LSTM-based sequential recommendations
    - Transformer-based sequential recommendations
    - Ensemble (weighted combination of both)
    - Genre-based filtering
    - Diversity-aware re-ranking
    """

    def __init__(self, lstm, transformer, window=5):
        self.lstm        = lstm
        self.transformer = transformer
        self.window      = window

    def _encode(self, titles):
        return np.array([[title_to_id[t] for t in titles]])

    def _get_preds(self, titles, model_choice='Ensemble',
                   lstm_weight=0.4, trans_weight=0.6):
        seq = self._encode(titles)
        if model_choice == 'LSTM':
            return self.lstm.predict(seq, verbose=0)[0]
        elif model_choice == 'Transformer':
            return self.transformer.predict(seq, verbose=0)[0]
        else:  # Ensemble
            p_l = self.lstm.predict(seq, verbose=0)[0]
            p_t = self.transformer.predict(seq, verbose=0)[0]
            return lstm_weight * p_l + trans_weight * p_t

    def recommend(self, titles, model_choice='Ensemble',
                  top_n=10, genre_filter=None,
                  diversity=False, exclude_seen=True):
        """
        Generate personalised recommendations.

        Parameters
        ----------
        titles       : list of 5 movie titles the user liked
        model_choice : 'LSTM' | 'Transformer' | 'Ensemble'
        top_n        : number of results to return
        genre_filter : optional genre string to filter by
        diversity    : if True, apply MMR diversity re-ranking
        exclude_seen : exclude input movies from results
        """
        preds = self._get_preds(titles, model_choice)

        # Exclude seen movies
        if exclude_seen:
            for t in titles:
                mid = title_to_id.get(t)
                if mid and mid < len(preds):
                    preds[mid] = 0.0

        # Genre filter
        if genre_filter and genre_filter != 'All':
            genre_mask = np.zeros(len(preds))
            for mid, genres in id_to_genres.items():
                if genre_filter in genres and mid < len(preds):
                    genre_mask[mid] = 1.0
            preds = preds * genre_mask

        # Get top candidates
        top_ids = np.argsort(preds)[-top_n*3:][::-1]

        results = []
        for mid in top_ids:
            title  = id_to_title.get(int(mid))
            genres = id_to_genres.get(int(mid), [])
            if title:
                results.append({
                    'id':     int(mid),
                    'title':  title,
                    'genres': genres,
                    'score':  float(preds[mid]),
                    'confidence': 'High' if preds[mid]>0.01
                                 else ('Medium' if preds[mid]>0.001 else 'Low')
                })
            if len(results) >= top_n:
                break

        return results[:top_n]

    def similar_movies(self, title, top_n=8):
        """Find movies similar to a given title using embedding cosine similarity."""
        mid = title_to_id.get(title)
        if not mid:
            return []
        emb_layer = self.transformer.get_layer('item_embedding')
        weights   = emb_layer.get_weights()[0]
        if mid >= len(weights):
            return []
        target = weights[mid]
        # Cosine similarity
        norms  = np.linalg.norm(weights, axis=1, keepdims=True) + 1e-9
        sims   = (weights @ target) / (norms.squeeze() * np.linalg.norm(target) + 1e-9)
        sims[mid] = -1  # exclude self
        top_ids = np.argsort(sims)[-top_n:][::-1]
        return [{
            'id':       int(i),
            'title':    id_to_title.get(int(i), 'Unknown'),
            'genres':   id_to_genres.get(int(i), []),
            'similarity': float(sims[i])
        } for i in top_ids if id_to_title.get(int(i))]

engine = RecommendationEngine(lstm_model, transformer_model, window=WINDOW)

# ── Quick sanity check ─────────────────────────────────────────
test_movies = all_titles[:5]
recs = engine.recommend(test_movies, model_choice='Ensemble', top_n=5)
print('\n🎬 Sample recommendations (Ensemble):')
print('-'*50)
for i, r in enumerate(recs, 1):
    print(f'  {i}. {r["title"]}')
    print(f'     Genres: {", ".join(r["genres"])}')
    print(f'     Score:  {r["score"]:.6f}  Confidence: {r["confidence"]}')
print('\n✅ Recommendation engine ready.')


## 🎨 Module 9 — Professional Gradio UI

In [ ]:
!pip install gradio>=4.0.0 -q
import gradio as gr
import datetime

# ── Safety: rebuild lookups if earlier cells weren't run ──────
import pandas as _pd, numpy as _np
try:
    _ = all_titles
except NameError:
    try:
        title_to_id  = _pd.Series(movies.movie_id.values, index=movies.title).to_dict()
        id_to_title  = _pd.Series(movies.title.values,    index=movies.movie_id.values).to_dict()
        id_to_genres = {}
        for _, _row in movies.iterrows():
            id_to_genres[_row['movie_id']] = [g for g in genre_cols[1:] if _row[g]==1]
        all_titles = sorted(movies['title'].unique().tolist())
        genre_list = genre_cols[1:]
    except Exception as _e:
        all_titles = []; genre_list = []
        print(f"⚠️  Run Modules 1-8 first: {_e}")

try:
    _ = engine
except NameError:
    try: engine = RecommendationEngine(lstm_model, transformer_model, window=WINDOW)
    except: pass

try:    _ = metrics
except: metrics = {'LSTM':{'HR10':0.0,'NDCG10':0.0},'Transformer':{'HR10':0.0,'NDCG10':0.0}}
try:    _ = lstm_acc
except: lstm_acc = lstm_top5 = lstm_top10 = lstm_time = 0.0
try:    _ = trans_acc
except: trans_acc = trans_top5 = trans_top10 = trans_time = 0.0
try:    _ = winner_model
except: winner_model = 'N/A'


# ── User database (demo accounts) ─────────────────────────────
USER_DB = {
    "durjoy": {"password": "123", "name": "Durjoy", "avatar": "🎬"},
}

# ── Session state ──────────────────────────────────────────────
session = {
    "logged_in": False,
    "username":  "",
    "name":      "",
    "avatar":    "",
    "search_history": [],    # list of dicts: {movies, model, genre, recs, timestamp}
    "watch_history": [],     # list of movie titles "opened"
}

# ── Genre emoji map ────────────────────────────────────────────
GENRE_EMOJI = {
    "Action":"💥","Adventure":"🗺️","Animation":"🎨","Children":"🧒",
    "Comedy":"😂","Crime":"🔫","Documentary":"📽️","Drama":"🎭",
    "Fantasy":"🧙","Film-Noir":"🕵️","Horror":"👻","Musical":"🎵",
    "Mystery":"🔍","Romance":"❤️","Sci-Fi":"🚀","Thriller":"😱",
    "War":"⚔️","Western":"🤠","unknown":"❓"
}

# ── TMDB-style movie card (markdown) ──────────────────────────
def movie_card_md(r, rank=None, show_score=True):
    genres = r.get("genres", [])
    genre_tags = "  ".join(f"`{GENRE_EMOJI.get(g,'🎬')} {g}`" for g in genres[:3])
    conf = r.get("confidence","")
    dot = {"High":"🟢","Medium":"🟡","Low":"🔴"}.get(conf,"⚪")
    score_line = f"\n> Score `{r['score']:.5f}`  {dot} {conf}" if show_score else ""
    num = f"**#{rank}**  " if rank else ""
    return f"""{num}### 🎬 {r['title']}
{genre_tags}{score_line}
"""

# ── Format movie info page ─────────────────────────────────────
def movie_info_page(title):
    if not title:
        return "⚠️ Please select a movie first."
    mid = title_to_id.get(title)
    genres = id_to_genres.get(mid, []) if mid else []
    genre_tags = "  ".join(f"`{GENRE_EMOJI.get(g,'🎬')} {g}`" for g in genres)

    # Get movie row
    mrow = movies[movies['title'] == title]
    year = ""
    if not mrow.empty:
        rd = mrow.iloc[0].get("release_date","")
        if rd and str(rd) != "nan":
            try: year = f"({str(rd).split('-')[0]})"
            except: pass

    # Rating stats
    mstats = ratings[ratings['movie_id'] == mid]['rating'] if mid else None
    avg_r = f"{mstats.mean():.2f} ⭐" if mstats is not None and len(mstats)>0 else "N/A"
    n_r   = f"{len(mstats):,}" if mstats is not None else "0"

    # Similar movies
    sims = engine.similar_movies(title, top_n=5)
    sim_lines = "\n".join(f"- **{s['title']}**  {'  '.join(f'`{g}`' for g in s['genres'][:2])}" for s in sims)

    # Log to watch history
    if session["logged_in"] and title not in session["watch_history"]:
        session["watch_history"].insert(0, title)
        session["watch_history"] = session["watch_history"][:20]

    return f"""# 🎬 {title} {year}

---

**Genres:** {genre_tags}

| Metric | Value |
|---|---|
| 🌟 Avg Rating | {avg_r} |
| 📊 Total Ratings | {n_r} |
| 🎭 Genres | {', '.join(genres) or 'Unknown'} |
| 🆔 Movie ID | {mid or 'N/A'} |

---

### 🔍 Similar Movies (AI-powered)
{sim_lines or '_No similar movies found._'}

---
*Data from MovieLens 100K · Similarity powered by CineAI Transformer embeddings*
"""

# ── Login function ─────────────────────────────────────────────
def do_login(username, password):
    u = username.strip().lower()
    if u in USER_DB and USER_DB[u]["password"] == password:
        session["logged_in"] = True
        session["username"]  = u
        session["name"]      = USER_DB[u]["name"]
        session["avatar"]    = USER_DB[u]["avatar"]
        session["search_history"] = []
        session["watch_history"]  = []
        ts = datetime.datetime.now().strftime("%H:%M")
        return (
            gr.update(visible=False),   # login_panel hidden
            gr.update(visible=True),    # main_app visible
            f"### {session['avatar']} Welcome back, **{session['name']}**!  `{ts}`",
            "",
        )
    return (
        gr.update(visible=True),
        gr.update(visible=False),
        "",
        "❌ Invalid username or password. Please check your credentials.",
    )

def do_logout():
    session.update({"logged_in":False,"username":"","name":"","avatar":"",
                    "search_history":[],"watch_history":[]})
    return (
        gr.update(visible=True),
        gr.update(visible=False),
        "",
    )

# ── Recommendation with history logging ───────────────────────
def tab_recommend(m1,m2,m3,m4,m5,model,top_n,genre):
    titles = [m1,m2,m3,m4,m5]
    if not all(titles):
        return "⚠️ Please select all 5 movies.", ""
    try:
        recs = engine.recommend(
            titles, model_choice=model,
            top_n=int(top_n), genre_filter=genre if genre!="All" else None
        )
        # Log to search history
        if session["logged_in"]:
            entry = {
                "timestamp": datetime.datetime.now().strftime("%d %b %H:%M"),
                "movies":    titles,
                "model":     model,
                "genre":     genre,
                "recs":      [r["title"] for r in recs[:3]],
            }
            session["search_history"].insert(0, entry)
            session["search_history"] = session["search_history"][:15]

        cards = []
        for i, r in enumerate(recs, 1):
            cards.append(movie_card_md(r, rank=i))
        result = "\n\n---\n\n".join(cards)

        summary = (
            f"**Model:** `{model}`  ·  **Results:** `{len(recs)}`  ·  "
            f"**Genre:** `{genre}`\n\n"
            f"LSTM `{lstm_acc:.4f}` · Transformer `{trans_acc:.4f}` · "
            f"Winner: **{winner_model}** 🏆"
        )
        return result, summary
    except Exception as e:
        return f"❌ Error: {str(e)}", ""

# ── Search History ─────────────────────────────────────────────
def get_search_history():
    if not session["logged_in"]:
        return "🔒 Please log in to view your history."
    if not session["search_history"]:
        return "📭 No searches yet. Go get some recommendations!"
    lines = [f"## 🕐 Search History — {session['name']}\n"]
    for i, e in enumerate(session["search_history"], 1):
        movies_str = " · ".join(f"`{m[:25]}`" for m in e["movies"])
        recs_str   = ", ".join(f"**{r}**" for r in e["recs"])
        lines.append(
            f"### {i}. {e['timestamp']}  `{e['model']}`  `{e['genre']}`\n"
            f"**You liked:** {movies_str}\n\n"
            f"**Top picks:** {recs_str}\n"
        )
    return "\n\n---\n\n".join(lines)

def get_watch_history():
    if not session["logged_in"]:
        return "🔒 Please log in."
    if not session["watch_history"]:
        return "📭 No movies browsed yet. Open a movie's info page!"
    lines = [f"## 👁️ Recently Browsed — {session['name']}\n"]
    for t in session["watch_history"]:
        mid = title_to_id.get(t)
        genres = id_to_genres.get(mid,[]) if mid else []
        g = "  ".join(f"`{GENRE_EMOJI.get(g,'🎬')} {g}`" for g in genres[:3])
        lines.append(f"- **{t}**  {g}")
    return "\n".join(lines)

# ── Similar movies ─────────────────────────────────────────────
def tab_similar(title):
    if not title: return "⚠️ Please select a movie."
    try:
        sims = engine.similar_movies(title, top_n=8)
        if not sims: return "⚠️ No similar movies found."
        cards = [f"## 🔍 Movies Similar to *{title}*\n"]
        for s in sims:
            genres = "  ".join(f"`{GENRE_EMOJI.get(g,'🎬')} {g}`" for g in s["genres"][:3])
            bar = "█" * int(s["similarity"]*20)
            cards.append(f"**{s['title']}**\n{genres}\nSimilarity: `{bar}` `{s['similarity']:.4f}`\n")
        return "\n\n---\n\n".join(cards)
    except Exception as e:
        return f"❌ Error: {str(e)}"

# ── Genre browse ───────────────────────────────────────────────
def tab_browse(genre):
    result = engine.recommend(all_titles[:5], model_choice="Ensemble", top_n=12, genre_filter=genre)
    if not result: return f"⚠️ No {genre} movies found."
    cards = [f"## {GENRE_EMOJI.get(genre,'🎬')} Top {genre} Movies\n"]
    for r in result:
        genres = "  ".join(f"`{GENRE_EMOJI.get(g,'🎬')} {g}`" for g in r["genres"][:3])
        cards.append(f"**{r['title']}**\n{genres}\n")
    return "\n\n---\n\n".join(cards)

# ── Analytics ──────────────────────────────────────────────────
def tab_analytics():
    def _s(val, fmt=".4f"):
        try: return format(val, fmt)
        except: return "N/A"
    lines = [
        "## 📊 CineAI Model Analytics Report\n",
        "### 🗄️ Dataset",
        f"| Stat | Value |",
        f"|---|---|",
        f"| Movies | {len(movies):,} |",
        f"| Users | {len(users):,} |",
        f"| Ratings | {len(ratings):,} |",
        f"| Train sequences | {len(X_train_s):,} |",
        f"| Test sequences | {len(X_test_s):,} |\n",
        "### 🧠 LSTM",
        f"| Metric | Score |",
        f"|---|---|",
        f"| Parameters | {lstm_model.count_params():,} |",
        f"| Accuracy | {lstm_acc:.4f} |",
        f"| Top-5 Acc | {lstm_top5:.4f} |",
        f"| Top-10 Acc | {lstm_top10:.4f} |",
        f"| HR@10 | {metrics['LSTM']['HR10']:.4f} |",
        f"| NDCG@10 | {metrics['LSTM']['NDCG10']:.4f} |",
        f"| Train time | {lstm_time/60:.1f} min |\n",
        "### ⚡ Transformer",
        f"| Metric | Score |",
        f"|---|---|",
        f"| Parameters | {transformer_model.count_params():,} |",
        f"| Accuracy | {trans_acc:.4f} |",
        f"| Top-5 Acc | {trans_top5:.4f} |",
        f"| Top-10 Acc | {trans_top10:.4f} |",
        f"| HR@10 | {metrics['Transformer']['HR10']:.4f} |",
        f"| NDCG@10 | {metrics['Transformer']['NDCG10']:.4f} |",
        f"| Train time | {trans_time/60:.1f} min |\n",
        f"### 🏆 Winner: **{winner_model}**",
    ]
    return "\n".join(lines)

# ═══════════════════════════════════════════════════════════════
# CUSTOM CSS — Cinematic Dark Theme  (consistent everywhere)
# ═══════════════════════════════════════════════════════════════
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600;700&display=swap');

/* ── Root: clean light cinema theme ── */
:root {
  --bg:       #f0f2f8;
  --surface:  #ffffff;
  --card:     #f7f8fc;
  --border:   #dde2ee;
  --accent:   #c0392b;
  --accent2:  #e67e22;
  --text:     #1a1a2e;
  --muted:    #6b7280;
  --radius:   12px;
}

/* ── Base — force light everywhere ── */
*, *::before, *::after { box-sizing: border-box; }
html, body,
.gradio-container,
.gradio-container > .main,
.gradio-container > .main > .wrap,
.app, footer { background: var(--bg) !important; color: var(--text) !important; }

/* ── Every block/panel/card ── */
.block, .gr-panel, .panel, .form, fieldset,
div.svelte-vt1mxs, div.gap, .padded, .bordered {
  background: var(--surface) !important;
  border: 1px solid var(--border) !important;
  border-radius: var(--radius) !important;
  box-shadow: 0 1px 6px rgba(0,0,0,0.06) !important;
}

/* ── Tabs ── */
.tabs > .tab-nav { border-bottom: 2px solid var(--border) !important; background: var(--surface) !important; }
.tab-nav button {
  color: var(--muted) !important; font-weight: 500 !important;
  background: transparent !important; border: none !important;
  border-bottom: 2px solid transparent !important;
  font-size: 0.88rem !important; padding: 10px 18px !important;
}
.tab-nav button:hover { color: var(--text) !important; }
.tab-nav button.selected {
  color: var(--accent) !important;
  border-bottom: 2px solid var(--accent) !important;
}

/* ── Inputs ── */
input, select, textarea,
input[type=text], input[type=password],
.gr-input, span[data-testid] input {
  background: var(--card) !important;
  border: 1.5px solid var(--border) !important;
  color: var(--text) !important;
  border-radius: 8px !important;
  font-family: 'DM Sans', sans-serif !important;
}
input:focus, select:focus, textarea:focus {
  border-color: var(--accent) !important;
  box-shadow: 0 0 0 3px rgba(192,57,43,0.12) !important;
  outline: none !important;
}

/* ── Labels & text ── */
label, .gr-label { color: var(--muted) !important; font-size: 0.78rem !important; font-weight: 600 !important; text-transform: uppercase !important; letter-spacing: 0.06em !important; }
p, li, span { color: var(--text) !important; }
h1, h2, h3, h4 { color: var(--text) !important; }

/* ── Markdown ── */
.prose p, .prose li { color: var(--text) !important; }
.prose h1,.prose h2,.prose h3 { color: var(--text) !important; border-bottom: 1px solid var(--border); padding-bottom: 6px; }
.prose code, code { background: #fef3f2 !important; color: var(--accent) !important; border: 1px solid #fbd5d2 !important; border-radius: 5px !important; padding: 1px 6px !important; font-size: 0.85em !important; }
.prose strong, strong { color: var(--accent2) !important; }
.prose table { border-collapse: collapse !important; width: 100% !important; }
.prose th { background: var(--card) !important; color: var(--muted) !important; padding: 8px 14px !important; border-bottom: 2px solid var(--border) !important; text-align: left !important; }
.prose td { border-top: 1px solid var(--border) !important; padding: 8px 14px !important; color: var(--text) !important; }
blockquote { border-left: 3px solid var(--accent) !important; padding-left: 14px !important; color: var(--muted) !important; margin: 8px 0 !important; }
hr { border-color: var(--border) !important; }

/* ── Primary button ── */
button.primary, .gr-button-primary, button[variant=primary] {
  background: linear-gradient(135deg, #e74c3c, #c0392b) !important;
  color: #fff !important; border: none !important;
  border-radius: 9px !important; font-weight: 600 !important;
  font-size: 0.9rem !important; padding: 10px 22px !important;
  box-shadow: 0 3px 12px rgba(192,57,43,0.3) !important;
  transition: all 0.2s ease !important;
}
button.primary:hover { transform: translateY(-1px) !important; box-shadow: 0 5px 18px rgba(192,57,43,0.45) !important; }

/* ── Secondary button ── */
button.secondary, .gr-button-secondary {
  background: var(--surface) !important; color: var(--text) !important;
  border: 1.5px solid var(--border) !important; border-radius: 9px !important;
}

/* ── Dropdown popup ── */
ul.options, .gr-dropdown, [data-testid=dropdown-options] {
  background: var(--surface) !important;
  border: 1px solid var(--border) !important;
  color: var(--text) !important;
}

/* ── Slider ── */
input[type=range] { accent-color: var(--accent) !important; }

/* ── Radio ── */
input[type=radio] { accent-color: var(--accent) !important; }

/* ── Login box ── */
#login-box {
  max-width: 430px; margin: 60px auto;
  background: var(--surface) !important;
  border: 1px solid var(--border) !important;
  border-radius: 20px !important; padding: 40px !important;
  box-shadow: 0 8px 40px rgba(0,0,0,0.10) !important;
}

/* ── Scrollbar ── */
::-webkit-scrollbar { width: 5px; }
::-webkit-scrollbar-track { background: var(--bg); }
::-webkit-scrollbar-thumb { background: var(--border); border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: var(--accent); }
"""

# ═══════════════════════════════════════════════════════════════
# BUILD GRADIO APP
# ═══════════════════════════════════════════════════════════════
with gr.Blocks(css=CUSTOM_CSS, title="🎬 CineAI — Movie Recommender") as demo:

    # ── LOGIN PANEL ──────────────────────────────────────────────
    with gr.Column(visible=True, elem_id="login-box") as login_panel:
        gr.Markdown("""
# 🎬 CineAI
### Intelligent Movie Recommendation System
---
*Deep Learning · LSTM · Transformer · MovieLens 100K*
        """)
        login_user  = gr.Textbox(label="Username", placeholder="Enter your username", max_lines=1)
        login_pass  = gr.Textbox(label="Password", placeholder="Password", type="password", max_lines=1)
        login_btn   = gr.Button("🔐 Sign In", variant="primary")
        login_err   = gr.Markdown("")
        gr.Markdown("*Enter your credentials to access CineAI*")

    # ── MAIN APP (hidden until login) ───────────────────────────
    with gr.Column(visible=False) as main_app:

        # ── Navbar ──
        with gr.Row(elem_id="navbar"):
            welcome_md = gr.Markdown("### 🎬 CineAI")
            with gr.Column(scale=0, min_width=130):
                logout_btn = gr.Button("🚪 Sign Out", variant="secondary")

        gr.Markdown("---")

        with gr.Tabs():

            # ── TAB 1: Personalized Recommendations ──────────────
            with gr.Tab("🎯 For You"):
                gr.Markdown("Pick **5 movies** you love — CineAI predicts what you'll watch next.")
                with gr.Row():
                    with gr.Column(scale=1):
                        m1 = gr.Dropdown(all_titles, label="Movie 1 — Oldest", filterable=True)
                        m2 = gr.Dropdown(all_titles, label="Movie 2", filterable=True)
                        m3 = gr.Dropdown(all_titles, label="Movie 3", filterable=True)
                        m4 = gr.Dropdown(all_titles, label="Movie 4", filterable=True)
                        m5 = gr.Dropdown(all_titles, label="Movie 5 — Most Recent", filterable=True)
                        with gr.Row():
                            model_dd = gr.Dropdown(["Ensemble","LSTM","Transformer"], value="Ensemble", label="Model")
                            genre_dd = gr.Dropdown(["All"]+genre_list, value="All", label="Genre Filter")
                        top_n_sl = gr.Slider(3, 15, value=8, step=1, label="Number of Results")
                        rec_btn  = gr.Button("🔍 Get Recommendations", variant="primary")
                    with gr.Column(scale=2):
                        summary_box = gr.Markdown()
                        result_box  = gr.Markdown()
                rec_btn.click(tab_recommend,
                    inputs=[m1,m2,m3,m4,m5,model_dd,top_n_sl,genre_dd],
                    outputs=[result_box, summary_box])

            # ── TAB 2: Movie Info Page ────────────────────────────
            with gr.Tab("🎬 Movie Info"):
                gr.Markdown("Browse detailed info for any movie — genres, ratings, AI-matched similar titles.")
                with gr.Row():
                    with gr.Column(scale=1):
                        info_dd  = gr.Dropdown(all_titles, label="Search Movie", filterable=True)
                        info_btn = gr.Button("📖 View Info", variant="primary")
                    with gr.Column(scale=2):
                        info_out = gr.Markdown("_Select a movie on the left to see its full info page._")
                info_btn.click(movie_info_page, inputs=info_dd, outputs=info_out)

            # ── TAB 3: Similar Movies ──────────────────────────────
            with gr.Tab("🔍 Find Similar"):
                gr.Markdown("Find movies similar to one you already love — powered by AI embeddings.")
                with gr.Row():
                    with gr.Column(scale=1):
                        sim_dd  = gr.Dropdown(all_titles, label="Select a Movie", filterable=True)
                        sim_btn = gr.Button("Find Similar", variant="primary")
                    with gr.Column(scale=2):
                        sim_out = gr.Markdown()
                sim_btn.click(tab_similar, inputs=sim_dd, outputs=sim_out)

            # ── TAB 4: Browse by Genre ─────────────────────────────
            with gr.Tab("🗂️ Browse Genres"):
                gr.Markdown("Explore AI-curated picks for every genre.")
                with gr.Row():
                    with gr.Column(scale=1):
                        genre_radio = gr.Radio(genre_list, label="Select Genre", value="Action")
                        browse_btn  = gr.Button("Browse", variant="primary")
                    with gr.Column(scale=2):
                        browse_out = gr.Markdown()
                browse_btn.click(tab_browse, inputs=genre_radio, outputs=browse_out)

            # ── TAB 5: Search History ──────────────────────────────
            with gr.Tab("🕐 Search History"):
                with gr.Row():
                    hist_btn   = gr.Button("🔄 Load Search History", variant="primary")
                    watch_btn  = gr.Button("👁️ Load Watch History", variant="secondary")
                history_out = gr.Markdown("_Click a button above to load your history._")
                hist_btn.click(get_search_history,  outputs=history_out)
                watch_btn.click(get_watch_history, outputs=history_out)

            # ── TAB 6: Model Analytics ─────────────────────────────
            with gr.Tab("📊 Analytics"):
                gr.Markdown("Full model benchmarking report.")
                analytics_btn = gr.Button("Load Report", variant="primary")
                analytics_out = gr.Markdown()
                analytics_btn.click(tab_analytics, outputs=analytics_out)

        gr.Markdown("---\n*CineAI · TensorFlow · Gradio · MovieLens 100K · Deep Learning Project*")

    # ── Wire login / logout ──────────────────────────────────────
    login_btn.click(
        do_login,
        inputs=[login_user, login_pass],
        outputs=[login_panel, main_app, welcome_md, login_err]
    )
    logout_btn.click(
        do_logout,
        outputs=[login_panel, main_app, welcome_md]
    )

# ── Launch ───────────────────────────────────────────────────────
demo.launch(share=True, show_error=True)
print("\n✅ CineAI Pro is live! Share the URL above with your faculty.")
